# Extraction Cost

Token usage and cost of the extraction stage, reconstructed because the API run logged no usage (Appendix A-D). Input tokens: the notebook-1 prompt rebuilt for each of the 52,947 documents and tokenised with the Llama 3.1 tokenizer (`unsloth/Meta-Llama-3.1-8B-Instruct`, local cache, chat-template tokens included). Output tokens: the JSON response rebuilt from the published keyword lists, calibrated on the raw responses kept for the 182-document sample and 1,999 Inspec documents. Cost: OpenRouter list price fetched at run time (`pricing_snapshot.json`; falls back to the archived snapshot, then to the manuscript's prices) and the model-page price. Time: lower bound from the 0.6 s pause between calls.

The run also sent 183 rows that were discarded at parsing; they are counted separately and excluded from the totals. Inputs: `EID_KEYWORDS.xlsx`, `data/insumo_row_to_eid.csv`, the raw responses in `data/keywords_llm_llama-3.1-8b-EN.csv` and `inspec_llama-3.1-8b-EN.csv`, and the private record file from `FTTS_PRIVATE_DIR`. Outputs in `results/e7_extraction_cost/`. Gate: real over reconstructed output tokens within [0.8, 1.25] on the sample.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"            # several notebooks run concurrently on this machine
os.environ["RAYON_NUM_THREADS"] = "4"          # thread pool of the fast tokenizer
os.environ["HF_HUB_OFFLINE"] = "1"             # tokenizer read from the local Hugging Face cache, nothing is downloaded
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import sys
sys.path.insert(0, "scripts")

import json
import urllib.request
from pathlib import Path
from string import Template

import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import TqdmWarning

torch.set_num_threads(4)
warnings.filterwarnings("ignore", category=TqdmWarning)   # no ipywidgets in this kernel: plain-text progress output

import common as C
import inspec_evaluation as IE

OUT = C.RESULTS / "e7_extraction_cost"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_ID = "meta-llama/llama-3.1-8b-instruct"
TOKENIZER = "unsloth/Meta-Llama-3.1-8B-Instruct"   # ungated copy of the Llama 3.1 tokenizer
SLEEP_BETWEEN_CALLS = 0.6                          # notebook 1: SLEEP_ENTRE_CALLS
MAX_ATTEMPTS = 4                                   # notebook 1: MAX_INTENTOS
MAX_OUTPUT_TOKENS = 700                            # notebook 1: max_tokens
MANUSCRIPT_PRICES = {                              # Appendix A-D, USD per token
    "list_price_2026-09-09": {"usd_per_prompt_token": 5e-8, "usd_per_completion_token": 8e-8},
    "model_page": {"usd_per_prompt_token": 2e-8, "usd_per_completion_token": 4e-8},
}

INPUTS = {
    "insumo": C.PRIVATE_DIR / "corpus_insumo_DEFINITIVO.csv",
    "keywords": C.REPO_ROOT / "EID_KEYWORDS.xlsx",
    "alignment": C.DATA / "insumo_row_to_eid.csv",
    "sample_8b": C.DATA / "keywords_llm_llama-3.1-8b-EN.csv",
    "inspec_8b": C.REPO_ROOT / "inspec_llama-3.1-8b-EN.csv",
    "inspec": C.REPO_ROOT / "dataset_inspec.csv",
}
missing = [k for k, p in INPUTS.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"missing inputs {missing}; the private records are read from FTTS_PRIVATE_DIR={C.PRIVATE_DIR}")
for k, p in INPUTS.items():
    print(f"{k:10s} {p}")
print("truncate_chars:", C.TRUNCATE_CHARS, "| results dir:", OUT)

insumo     /home/mat/academic-writing/papers/from_text_to_structure/corpus_insumo_DEFINITIVO.csv
keywords   /home/mat/academic-writing/papers/from_text_to_structure/data_repo/EID_KEYWORDS.xlsx
alignment  /home/mat/academic-writing/papers/from_text_to_structure/data_repo/data/insumo_row_to_eid.csv
sample_8b  /home/mat/academic-writing/papers/from_text_to_structure/data_repo/data/keywords_llm_llama-3.1-8b-EN.csv
inspec_8b  /home/mat/academic-writing/papers/from_text_to_structure/data_repo/inspec_llama-3.1-8b-EN.csv
inspec     /home/mat/academic-writing/papers/from_text_to_structure/data_repo/dataset_inspec.csv
truncate_chars: 3000 | results dir: /home/mat/academic-writing/papers/from_text_to_structure/data_repo/results/e7_extraction_cost


In [2]:
# ============================================================
# PROMPT OF THE ORIGINAL RUN (verbatim from `1. LLMS.ipynb`, cell 0)
# ============================================================
SYSTEM_MSG = (
    'Respond ONLY with a valid JSON in the form {"keywords":[...]}. '
    'Do not include explanations, markdown, apologies, or code blocks. '
    'STRICT LANGUAGE RULE: '
    '- The output MUST contain ONLY ENGLISH WORDS. '
    '- Translate foreign terms into English; if translation is impossible, use a English descriptive equivalent. '
    '- Before responding, internally verify that EVERY keyword is fully in English. '
)
PROMPT_TMPL = Template("""
You are an expert assistant in bibliographic analysis.
From the following article record (authors, title, year, journal, abstract, original keywords),
extract EXACTLY 5 terms that represent the main concepts of the article.

Rules:
- ALWAYS return: {"keywords": ["term1", "term2", ...]}.
- Output MUST be ONLY in ENGLISH; no other languages under any circumstance.
- Normalize to lowercase, without accents or special characters (ej. "mathematics education").
- Accept short multiword phrases (2–4 words).
- Avoid generic terms such as: "article", "study", "analysis", "research", "work", "keyword".
- Prioritize disciplinary concepts, population, context, method, theory.
- ALWAYS return exactly 5 items in the list; if a concept cannot be expressed in English, use the string "null" as a placeholder, which still counts toward the five terms.

Article text:
$texto

Before responding, SELF-CHECK: "all the keywords are in English without exception?"
""")
print(SYSTEM_MSG)
print(PROMPT_TMPL.template)

Respond ONLY with a valid JSON in the form {"keywords":[...]}. Do not include explanations, markdown, apologies, or code blocks. STRICT LANGUAGE RULE: - The output MUST contain ONLY ENGLISH WORDS. - Translate foreign terms into English; if translation is impossible, use a English descriptive equivalent. - Before responding, internally verify that EVERY keyword is fully in English. 

You are an expert assistant in bibliographic analysis.
From the following article record (authors, title, year, journal, abstract, original keywords),
extract EXACTLY 5 terms that represent the main concepts of the article.

Rules:
- ALWAYS return: {"keywords": ["term1", "term2", ...]}.
- Output MUST be ONLY in ENGLISH; no other languages under any circumstance.
- Normalize to lowercase, without accents or special characters (ej. "mathematics education").
- Accept short multiword phrases (2–4 words).
- Avoid generic terms such as: "article", "study", "analysis", "research", "work", "keyword".
- Prioritize d

In [3]:
# ============================================================
# TOKENIZER, RUN METADATA AND HELPERS
# ============================================================
T = C.Timer()
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TOKENIZER)
meta = C.env_metadata(experiment="E7 extraction cost reconstruction", tokenizer=TOKENIZER,
                      tokenizer_vocab=len(tok),
                      inputs={k: {"path": str(p), "sha256": C.sha256(p)} for k, p in INPUTS.items()},
                      llm_calls=0, paid_api_calls=0)
print(json.dumps({k: v for k, v in meta.items() if k != "inputs"}, indent=1))
print(pd.DataFrame([{"input": k, **v} for k, v in meta["inputs"].items()]).to_string(index=False))


def prompt_tokens(texts):
    """Token count of the full chat prompt for each record text (batched)."""
    # Constant part: chat template + system + user template with an empty article text.
    base = tok.apply_chat_template([{"role": "system", "content": SYSTEM_MSG},
                                    {"role": "user", "content": PROMPT_TMPL.substitute(texto="")}],
                                   add_generation_prompt=True, tokenize=True)
    enc = tok(list(texts), add_special_tokens=False)["input_ids"]
    # Joining the article text inside the template can merge/split one token at each boundary.
    return np.array([len(base) + len(e) for e in enc]), len(base)


def reconstruct_response(kws) -> str:
    return json.dumps({"keywords": [str(k) for k in kws]}, ensure_ascii=False)


print("\nconstant part of the prompt (chat template + system + user template):", prompt_tokens([""])[1], "tokens")

{
 "started_utc": "2026-09-25T00:35:05.355480+00:00",
 "python": "3.12.3",
 "platform": "Linux-7.0.0-31-generic-x86_64-with-glibc2.39",
 "machine": "x86_64",
 "cpu_count": 16,
 "git_commit": "af664fad8d65723998851025c9d052e9bf1fa73e",
 "packages": {
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "scipy": "1.15.3",
  "sklearn": "1.9.0",
  "networkx": "3.6.1",
  "community": "0.16",
  "sentence_transformers": "5.1.1",
  "torch": "2.8.0+cpu",
  "transformers": "4.57.6",
  "langdetect": "unknown",
  "openai": "1.107.2"
 },
 "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
 "embedding_model_revision": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
 "experiment": "E7 extraction cost reconstruction",
 "tokenizer": "unsloth/Meta-Llama-3.1-8B-Instruct",
 "tokenizer_vocab": 128256,
 "llm_calls": 0,
 "paid_api_calls": 0
}
    input                                                                                                     path                                     

In [4]:
# ============================================================
# GATE: CALIBRATION OF THE OUTPUT RECONSTRUCTION ON THE SAVED RAW RESPONSES
# ============================================================
calib_rows, exact_rows, examples = [], [], []
for name, path in (("validation_sample_182", INPUTS["sample_8b"]), ("inspec_2000", INPUTS["inspec_8b"])):
    df = pd.read_csv(path)
    real = [str(x) for x in df["raw_response"] if isinstance(x, str)]
    recon = [reconstruct_response(IE.safe_parse_list(k)) for k, r in zip(df["keywords_llm"], df["raw_response"]) if isinstance(r, str)]
    ids_real = tok(real, add_special_tokens=False)["input_ids"]
    ids_recon = tok(recon, add_special_tokens=False)["input_ids"]
    n_real = np.array([len(x) for x in ids_real])
    n_recon = np.array([len(x) for x in ids_recon])
    exact_str = float(np.mean([a.strip() == b for a, b in zip(real, recon)]))
    calib_rows.append({"set": name, "responses": len(real), "mean_tokens_real": float(n_real.mean()),
                       "mean_tokens_reconstructed": float(n_recon.mean()),
                       "ratio_real_over_reconstructed": float(n_real.sum() / n_recon.sum()),
                       "share_string_identical": exact_str,
                       "max_tokens_real": int(n_real.max()), "p99_tokens_real": float(np.percentile(n_real, 99))})
    # Per-document exactness (printed for the manuscript check; the archived file layout is kept unchanged).
    exact_rows.append({"set": name, "responses": len(real),
                       "same_token_count": int((n_real == n_recon).sum()),
                       "same_token_sequence": int(sum(a == b for a, b in zip(ids_real, ids_recon))),
                       "string_identical_after_strip": int(sum(a.strip() == b for a, b in zip(real, recon))),
                       "tokens_real_minus_reconstructed": int(n_real.sum() - n_recon.sum())})
    for a, b, na, nb in zip(real, recon, n_real, n_recon):
        if a.strip() != b and len(examples) < 6:
            examples.append({"set": name, "tokens_real": int(na), "tokens_recon": int(nb),
                             "raw_response": a.strip()[:110], "reconstructed": b[:110]})
calib = pd.DataFrame(calib_rows)
calib.to_csv(OUT / "output_reconstruction_calibration.csv", index=False)
ratio = float(calib.loc[calib.set == "validation_sample_182", "ratio_real_over_reconstructed"].iloc[0])
gate_pass = bool(0.8 <= ratio <= 1.25)
exact_df = pd.DataFrame(exact_rows)

pd.set_option("display.width", 250)
print(calib.round(4).to_string(index=False))
print()
print(exact_df.to_string(index=False))
print(f"\nraw responses checked: {int(exact_df.responses.sum()):,} | same token count: {int(exact_df.same_token_count.sum()):,}"
      f" | same token sequence: {int(exact_df.same_token_sequence.sum()):,}"
      f" | string identical: {int(exact_df.string_identical_after_strip.sum()):,}")
if examples:
    print("\nexamples of responses that differ from their reconstruction (whitespace or quoting):")
    print(pd.DataFrame(examples).to_string(index=False))
print(f"\nGATE (0.8 <= ratio <= 1.25 on the validation sample): {'PASS' if gate_pass else 'FAIL'}   ratio = {ratio:.6f}")
print(f"elapsed {T.mark('calibration'):.1f} s")

                  set  responses  mean_tokens_real  mean_tokens_reconstructed  ratio_real_over_reconstructed  share_string_identical  max_tokens_real  p99_tokens_real
validation_sample_182        182           27.5110                    27.5110                            1.0                    1.00               35             34.0
          inspec_2000       1999           26.6853                    26.6863                            1.0                    0.98               40             36.0

                  set  responses  same_token_count  same_token_sequence  string_identical_after_strip  tokens_real_minus_reconstructed
validation_sample_182        182               182                  182                           182                                0
          inspec_2000       1999              1984                 1959                          1959                               -2

raw responses checked: 2,181 | same token count: 2,166 | same token sequence: 2,141 | string

In [5]:
# ============================================================
# INSPEC BENCHMARK RUN (2,000 calls, same prompt)
# ============================================================
insp = pd.read_csv(INPUTS["inspec"])
insp_tokens, base_len = prompt_tokens([str(t)[:C.TRUNCATE_CHARS] for t in insp["insumo"]])
insp_out = np.array([len(x) for x in tok([reconstruct_response(IE.safe_parse_list(k)) for k in pd.read_csv(INPUTS["inspec_8b"])["keywords_llm"]],
                                          add_special_tokens=False)["input_ids"]])
print(f"Inspec calls = {len(insp):,} | prompt constant = {base_len} tokens | input tokens = {int(insp_tokens.sum()):,} "
      f"(mean {insp_tokens.mean():.1f}, max {int(insp_tokens.max())}) | reconstructed output tokens = {int(insp_out.sum()):,}")
print(f"elapsed {T.mark('inspec_tokens'):.1f} s")

Inspec calls = 2,000 | prompt constant = 317 tokens | input tokens = 938,971 (mean 469.5, max 973) | reconstructed output tokens = 53,351
elapsed 3.4 s


In [6]:
# ============================================================
# FULL CORPUS (one call per record of the private file)
# ============================================================
ins = pd.read_csv(INPUTS["insumo"])["insumo"].astype(str)
align = C.load_alignment(INPUTS["alignment"])
pub = C.load_published_keywords(INPUTS["keywords"], notebook_semantics=False)
seen = [s[:C.TRUNCATE_CHARS] for s in ins]          # every record was sent once: input tokens for all rows
in_tokens, base_len = prompt_tokens(seen)
in_tokens_all = in_tokens.copy()
out_recon = [reconstruct_response(pub[e]) for e in align.eid]
out_tokens = np.array([len(x) for x in tok(out_recon, add_special_tokens=False)["input_ids"]])
# Records without a linked published output are charged the mean output length.
out_full = np.full(len(ins), out_tokens.mean())
out_full[align.insumo_row.to_numpy()] = out_tokens
out_full_all = out_full.copy()
eid_by_row = pd.Series(align.eid.values, index=align.insumo_row.values).reindex(range(len(ins)))
lengths = ins.str.len().to_numpy()

pub_rows = align.insumo_row.to_numpy()                # the 52,947 documents of the corpus (one row per EID)
in_corpus = np.zeros(len(ins), dtype=bool); in_corpus[pub_rows] = True
n_calls = int(in_corpus.sum())
print(f"records sent in the run = {len(ins):,} | in the corpus (published output) = {n_calls:,} | discarded at parsing (not in the corpus) = {len(ins) - n_calls:,}")
print(f"input tokens of the {len(ins) - n_calls} discarded rows (not counted below): {int(in_tokens[~in_corpus].sum()):,}")
in_tokens, out_full, lengths = in_tokens[in_corpus], out_full[in_corpus], lengths[in_corpus]
print(f"input tokens over the {n_calls:,} corpus documents: total = {int(in_tokens.sum()):,} | mean per call = {in_tokens.mean():.2f} | max = {int(in_tokens.max())} | prompt constant = {base_len}")
print(f"output tokens (reconstructed): total = {out_full.sum():,.1f} | mean per linked call = {out_tokens.mean():.3f}")
print(f"records longer than {C.TRUNCATE_CHARS:,} characters (truncated) = {int((lengths > C.TRUNCATE_CHARS).sum()):,} "
      f"({(lengths > C.TRUNCATE_CHARS).mean():.2%}) | record length: mean = {lengths.mean():.1f}, median = {np.median(lengths):.0f}")
print(pd.Series(in_tokens, name="input_tokens_per_call").describe().round(1).to_string())
print(f"elapsed {T.mark('corpus_tokens'):.1f} s")

records sent in the run = 53,130 | in the corpus (published output) = 52,947 | discarded at parsing (not in the corpus) = 183
input tokens of the 183 discarded rows (not counted below): 118,446
input tokens over the 52,947 corpus documents: total = 33,857,543 | mean per call = 639.46 | max = 1215 | prompt constant = 317
output tokens (reconstructed): total = 1,394,442.0 | mean per linked call = 26.337
records longer than 3,000 characters (truncated) = 2,008 (3.79%) | record length: mean = 1716.0, median = 1638
count    52947.0
mean       639.5
std        109.1
min        342.0
25%        567.0
50%        629.0
75%        700.0
max       1215.0
elapsed 17.7 s


In [7]:
# ============================================================
# LIST PRICE: live fetch, then the archived snapshot, then the manuscript's price points
# ============================================================
def fetch_pricing() -> dict:
    with urllib.request.urlopen("https://openrouter.ai/api/v1/models", timeout=30) as r:
        data = json.load(r)["data"]
    for m in data:
        if m["id"] == MODEL_ID:
            return {"model": MODEL_ID, "fetched_utc": C.now_utc(),
                    "usd_per_prompt_token": float(m["pricing"]["prompt"]),
                    "usd_per_completion_token": float(m["pricing"]["completion"]),
                    "context_length": m.get("context_length"), "source": "https://openrouter.ai/api/v1/models"}
    raise KeyError(f"{MODEL_ID} is not listed by OpenRouter")


snapshot_path = OUT / "pricing_snapshot.json"
archived_snapshot = json.loads(snapshot_path.read_text(encoding="utf-8")) if snapshot_path.exists() else None
try:
    pricing = fetch_pricing()
    price_source = "live OpenRouter API"
except Exception as exc:
    print("pricing fetch failed:", repr(exc))
    if archived_snapshot is not None:
        pricing = dict(archived_snapshot)
        pricing["fetched_utc"] = f"{archived_snapshot.get('fetched_utc')} (archived pricing_snapshot.json; live fetch failed on {C.now_utc()})"
        price_source = "archived pricing_snapshot.json"
    else:
        pricing = {"model": MODEL_ID,
                   "fetched_utc": f"2026-09-09 (manuscript Appendix A-D; live fetch failed on {C.now_utc()})",
                   **MANUSCRIPT_PRICES["list_price_2026-09-09"], "context_length": 131072,
                   "source": "https://openrouter.ai/api/v1/models"}
        price_source = "manuscript price points"
C.write_json(pricing, snapshot_path)
p_in, p_out = pricing["usd_per_prompt_token"], pricing["usd_per_completion_token"]
print(f"price source used: {price_source}")
print(f"USD per million tokens: input {p_in * 1e6:.3f} | output {p_out * 1e6:.3f} | fetched: {pricing['fetched_utc']}")
if archived_snapshot is not None:
    print(f"archived snapshot ({archived_snapshot.get('fetched_utc')}): input {archived_snapshot['usd_per_prompt_token'] * 1e6:.3f}"
          f" | output {archived_snapshot['usd_per_completion_token'] * 1e6:.3f}")
print(f"manuscript price points: list price 9 September 2026 = 0.05 / 0.08; model page = 0.02 / 0.04 (USD per million tokens)")

price source used: live OpenRouter API
USD per million tokens: input 0.050 | output 0.080 | fetched: 2026-09-25T00:35:20.327985+00:00
archived snapshot (2026-09-25T00:34:14.947698+00:00): input 0.050 | output 0.080
manuscript price points: list price 9 September 2026 = 0.05 / 0.08; model page = 0.02 / 0.04 (USD per million tokens)


In [8]:
# ============================================================
# COST, WALL-CLOCK BOUNDS AND OUTPUT FILES
# ============================================================
# n_calls = the 52,947 corpus documents (set above)
total_in = int(in_tokens.sum())
total_out_recon = float(out_full.sum())
total_out_cal = total_out_recon * ratio
cost = {
    "calls_minimum": n_calls,
    "input_tokens_total": total_in,
    "input_tokens_mean_per_call": float(in_tokens.mean()),
    "input_tokens_max_per_call": int(in_tokens.max()),
    "prompt_constant_tokens": int(base_len),
    "output_tokens_total_reconstructed": total_out_recon,
    "output_tokens_total_calibrated": total_out_cal,
    "output_tokens_mean_per_call_calibrated": total_out_cal / n_calls,
    "usd_input_at_list_price": total_in * p_in,
    "usd_output_at_list_price": total_out_cal * p_out,
    "usd_total_at_list_price": total_in * p_in + total_out_cal * p_out,
    "usd_per_1000_documents": (total_in * p_in + total_out_cal * p_out) / n_calls * 1000,
    "usd_total_if_every_call_retried_max_attempts": (total_in * p_in + total_out_cal * p_out) * MAX_ATTEMPTS,
    "wall_clock_lower_bound_hours_from_inter_call_delay": n_calls * SLEEP_BETWEEN_CALLS / 3600,
    "wall_clock_hours_if_mean_latency_1s": n_calls * (SLEEP_BETWEEN_CALLS + 1.0) / 3600,
    "wall_clock_hours_if_mean_latency_2s": n_calls * (SLEEP_BETWEEN_CALLS + 2.0) / 3600,
    "max_output_tokens_setting": MAX_OUTPUT_TOKENS,
    "records_truncated_at_3000_chars": int((lengths > C.TRUNCATE_CHARS).sum()),
    "share_records_truncated": float((lengths > C.TRUNCATE_CHARS).mean()),
    "record_chars_mean": float(lengths.mean()), "record_chars_median": float(np.median(lengths)),
}
inspec_cost = {"calls": int(len(insp)), "input_tokens_total": int(insp_tokens.sum()),
               "output_tokens_total_calibrated": float(insp_out.sum() * ratio),
               "usd_total_at_list_price": float(insp_tokens.sum() * p_in + insp_out.sum() * ratio * p_out)}
summary = {"model": MODEL_ID, "pricing": pricing, "output_calibration_ratio_used": ratio,
           "full_corpus": cost, "inspec_benchmark": inspec_cost,
           "notes": ["Token counts use the Llama 3.1 tokenizer with the chat template control tokens; "
                     "OpenRouter providers may add a few tokens of their own.",
                     "Prices are the list prices at fetch time; the original run took place in 2025.",
                     "Retries were not logged; the upper bound assumes every call used all 4 attempts."]}
C.write_json(summary, OUT / "summary.json")
pd.DataFrame({"row": np.arange(len(ins)), "eid": eid_by_row.values,
              "record_chars": ins.str.len().to_numpy(), "input_tokens": in_tokens_all, "in_corpus": in_corpus,
              "output_tokens_reconstructed": out_full_all}).to_csv(OUT / "per_document_tokens.csv.gz", index=False, compression="gzip")
meta.update({"completed_utc": C.now_utc(), "timings_seconds": T.marks, "status": "complete"})
C.write_json(meta, OUT / "metadata.json")
C.write_json({"gate": "output reconstruction calibrated on saved raw responses",
              "pass": gate_pass, "ratio_real_over_reconstructed": ratio,
              "calibration": calib.to_dict(orient="records")}, OUT / "validation.json")

# The same usage at the manuscript's second price point (model page: US$0.02 and US$0.04 per million tokens).
alt = MANUSCRIPT_PRICES["model_page"]
usd_model_page = total_in * alt["usd_per_prompt_token"] + total_out_cal * alt["usd_per_completion_token"]

print(json.dumps(cost, indent=1))
print(json.dumps(inspec_cost, indent=1))
print(f"\nUSD at the model-page price (0.02 / 0.04 per million tokens): {usd_model_page:.4f}")
print("timings (s):", T.marks)
print("files written:", sorted(p.name for p in OUT.iterdir()))

{
 "calls_minimum": 52947,
 "input_tokens_total": 33857543,
 "input_tokens_mean_per_call": 639.4610270648006,
 "input_tokens_max_per_call": 1215,
 "prompt_constant_tokens": 317,
 "output_tokens_total_reconstructed": 1394442.0,
 "output_tokens_total_calibrated": 1394442.0,
 "output_tokens_mean_per_call_calibrated": 26.33656297807241,
 "usd_input_at_list_price": 1.69287715,
 "usd_output_at_list_price": 0.11155536,
 "usd_total_at_list_price": 1.80443251,
 "usd_per_1000_documents": 0.03407997639148583,
 "usd_total_if_every_call_retried_max_attempts": 7.21773004,
 "wall_clock_lower_bound_hours_from_inter_call_delay": 8.824499999999999,
 "wall_clock_hours_if_mean_latency_1s": 23.532000000000004,
 "wall_clock_hours_if_mean_latency_2s": 38.23950000000001,
 "max_output_tokens_setting": 700,
 "records_truncated_at_3000_chars": 2008,
 "share_records_truncated": 0.03792471717000019,
 "record_chars_mean": 1716.0020208888134,
 "record_chars_median": 1638.0
}
{
 "calls": 2000,
 "input_tokens_total": 

## Check against the manuscript

In [9]:
# ============================================================
# CHECK AGAINST THE MANUSCRIPT (main.tex values transcribed by hand; nothing here edits the manuscript)
# ============================================================
def manuscript_table(rows):
    out = []
    for loc, qty, tex, val, nd in rows:
        try:
            tex_num = float(str(tex).replace(",", "").replace("%", ""))
        except ValueError:
            tex_num = None
        if val is None or (isinstance(val, float) and np.isnan(val)):
            comp, flag = "nan", "n/a"
        else:
            comp = f"{float(val):,.{nd}f}" if nd > 0 else f"{int(round(float(val))):,}"
            if tex_num is None:
                flag = "n/a (not in main.tex)"
            else:
                flag = "match" if abs(round(float(val), nd) - tex_num) < 1e-9 else "differs"
        out.append({"location": loc, "quantity": qty, "main.tex": str(tex), "computed": comp, "flag": flag})
    return pd.DataFrame(out)


APP = "Appendix A-D, Model and Inference Configuration"
LET = "Response letter, R2-5 (wording of 24 September 2026)"
n_raw = int(exact_df.responses.sum())
rows = [
    (APP, "input tokens (million)", "33.86", total_in / 1e6, 2),
    (APP, "output tokens (million)", "1.39", total_out_cal / 1e6, 2),
    (APP, "list price on 9 September 2026, USD per million input tokens", "0.05", p_in * 1e6, 2),
    (APP, "list price on 9 September 2026, USD per million output tokens", "0.08", p_out * 1e6, 2),
    (APP, "cost at the list price (USD)", "1.80", cost["usd_total_at_list_price"], 2),
    (APP, "cost at the model-page price, 0.02 / 0.04 per million (USD)", "0.73", usd_model_page, 2),
    (APP, "records exceeding the 3,000-character input limit", "2,008", cost["records_truncated_at_3000_chars"], 0),
    (APP, "share of records truncated (%)", "3.8", cost["share_records_truncated"] * 100, 1),
    (APP, "records over which usage is counted (one call each)", "52,947", n_calls, 0),
    (APP, "maximum output tokens per call", "700", MAX_OUTPUT_TOKENS, 0),
    (APP, "retry attempts per document", "4", MAX_ATTEMPTS, 0),
    (APP, "inter-call delay (s)", "0.6", SLEEP_BETWEEN_CALLS, 1),
    (APP, "input truncation (characters)", "3,000", C.TRUNCATE_CHARS, 0),
    (APP, "wall-clock lower bound from the inter-call delay (hours)", "not stated", cost["wall_clock_lower_bound_hours_from_inter_call_delay"], 2),
    (APP, "price source used in this run", "list price of 9 September 2026", None, 0),
    (LET, "documents with a raw response retained", "2,181", n_raw, 0),
    (LET, "documents whose reconstruction has the same token count", "2,166", int(exact_df.same_token_count.sum()), 0),
    (LET, "documents whose reconstruction has the same token sequence", "2,141", int(exact_df.same_token_sequence.sum()), 0),
    (LET, "documents whose reconstruction is string-identical", "2,141", int(exact_df.string_identical_after_strip.sum()), 0),
    (LET, "total output tokens, real minus reconstructed, on those documents (within 0.004%)", "-2", int(exact_df.tokens_real_minus_reconstructed.sum()), 0),
]
check = manuscript_table(rows)
check.loc[check.quantity == "price source used in this run", ["computed", "flag"]] = [price_source, "info"]
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 70)
print(check.to_string(index=False))
print(f"\n{(check.flag == 'match').sum()} match, {(check.flag == 'differs').sum()} differ, "
      f"{(~check.flag.isin(['match', 'differs'])).sum()} informational")

                                            location                                                                          quantity                       main.tex            computed                  flag
     Appendix A-D, Model and Inference Configuration                                                            input tokens (million)                          33.86               33.86                 match
     Appendix A-D, Model and Inference Configuration                                                           output tokens (million)                           1.39                1.39                 match
     Appendix A-D, Model and Inference Configuration                      list price on 9 September 2026, USD per million input tokens                           0.05                0.05                 match
     Appendix A-D, Model and Inference Configuration                     list price on 9 September 2026, USD per million output tokens                           0.08   